# 02. RAG 데모 (LangGraph + Text2Cypher + Langfuse)
`질문 → analyze_question → link_entities → generate_cypher → execute_query(재시도≤2) → generate_answer`
벡터 검색 대신 **그래프 질의**로 근거를 찾고, 답변에는 **근거 트리플·출처 URL**이 붙습니다. 로직은 `src/maplerag/rag/`.

**비용 안전장치**: `RUN_LLM = False` 가 기본입니다. 켜면 질문 1개당 모델 호출 약 3회(≈$0.02)가 발생합니다.

In [ ]:
RUN_LLM = False
QUESTION = "소울 어썰트 데미지 변화는?"
from maplerag.rag.guard import CypherRejected, guard

## 1. Cypher 안전장치 (모델 호출 없음)
읽기 전용·단일문·LIMIT 을 코드로 강제합니다. 모델이 쓰기 쿼리를 만들어도 실행 전에 막힙니다.

In [ ]:
for q in ["MATCH (n:maple__Skill) RETURN n.rdfs__label",
          "MATCH (n) DETACH DELETE n",
          "CALL apoc.cypher.doIt('x', {})",
          "MATCH (n) OPTIONAL MATCH (n)-[:r]->(m) WHERE n.x=1 RETURN n"]:
    try:
        print("허용 →", guard(q))
    except CypherRejected as e:
        print("차단 →", e, "|", q[:40])

## 2. 개체 연결 (모델 호출 없음)
임베딩 없이 라벨·별칭 부분일치로 후보를 찾습니다. 같은 이름의 노드가 여러 개면 `n_nodes` 로 알려 줍니다.

In [ ]:
from maplerag.rag.pipeline import link_entities
for c in link_entities({"analysis": {"entity_mentions": ["에르다 샤워", "유피테르"]}})["candidates"]:
    print(c["label"], c["classes"], "노드 수:", c["n_nodes"])

## 3. 질문 실행 (유료, 기본 꺼짐)
노드가 끝날 때마다 상태 변경분을 보여 주고, 마지막에 Langfuse 트레이스 링크를 출력합니다.

In [ ]:
if RUN_LLM:
    from maplerag.rag.service import ask_stream
    final, trace = {}, {}
    for node, upd in ask_stream(QUESTION, tags=["notebook"]):
        if node == "__trace__":
            trace = upd
            continue
        final.update(upd)
        print(f"[{node}]", {k: (v if k != "rows" else len(v)) for k, v in upd.items() if k in ("cypher", "last_error", "attempts", "status", "rows")})
    print("답변:", final["answer"])
    print("상태:", final["status"])
    for e in final.get("evidence", [])[:5]:
        print("  근거:", e["subject"], e["attribute"], e["before"], "→", e["object"], e["patch_date"], "|", e["evidence"])
    print("출처:", final["sources"])
    from langfuse import get_client
    print("트레이스:", get_client().get_trace_url(trace_id=trace["trace_id"]))
else:
    print("RUN_LLM=False: 모델 호출을 건너뜁니다. 터미널: uv run python -m maplerag.rag.run 질문")

## 4. 캐릭터 조회 경로 (LLM 없음)
닉네임 → Open API → **메모리 임시 트리플** → SPARQL. 영구 저장하지 않습니다. 응답에는 `Data based on NEXON Open API` 표기가 붙습니다.
응답 캐시가 없으면 Open API 를 최대 3회 호출합니다(개발 키: 초당 5회·일 1,000회).

In [ ]:
from maplerag.character.lookup import ask_character
r = ask_character("꽃게쥬", "스타포스 높은 장비는?")
print(r.answer)
print(f"임시 트리플 {r.triples}개 / 이번 실제 API 호출 {r.live_calls}회 / {r.attribution}")

## 5. 평가 (Langfuse 데이터셋)
질문 36개는 `eval/questions.jsonl` (코드와 분리). 정답이 그래프에 실제로 있는지 모델 없이 확인합니다. 실행은 터미널 `maplerag.evaluation.run_eval` (기본 3문항, 전체는 `--all --yes`).

In [ ]:
from collections import Counter
from maplerag.evaluation.dataset import fingerprint, gold_check, load_questions
qs = load_questions()
print("질문", len(qs), "개:", dict(Counter(q["category"] for q in qs)), "| 지문", fingerprint(qs))
r = gold_check()
print(f"정답 검증: {r['ok']}/{r['checked']} 통과", r["problems"])